In [30]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-5.6-luna"


def make_model():
    """이 단원이 쓰는 챗 모델.

    이 모델은 temperature 를 받지 않습니다(0 을 넘기면 400 이 옵니다). 그래서 출력을 고정할
    손잡이가 없고, 같은 입력에도 답이 흔들립니다(자동화 파이프라인 단원 6절이 그 흔들림을
    여러 번 돌려 잽니다).
    """
    return ChatOpenAI(model=MODEL_NAME)


print("모델:", MODEL_NAME)

모델: gpt-5.6-luna


In [31]:
# [제공 코드] 데이터 파일 읽기: 이 셀은 실행만 하세요.
import json
from pathlib import Path

# 교안은 단원 폴더에서, 정답 노트북은 정답/ 폴더에서 돌아가므로 두 경로를 모두 본다.
_DATA = _DATA = Path("../전처리")

def load_jsonl(name):
    """data/<name> 을 한 줄씩 읽어 dict 리스트로 돌려준다(한 줄에 JSON 하나)."""
    rows = []
    for line in (_DATA / name).read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def load_json(name):
    """data/<name> 을 통째로 읽어 dict 로 돌려준다(사전 파일용)."""
    return json.loads((_DATA / name).read_text(encoding="utf-8"))


print("데이터 폴더:", _DATA)

데이터 폴더: ..\전처리


In [32]:
from langchain_core.prompts import ChatPromptTemplate

few_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
너는 한국 요리 레시피 제목에서 표준 음식(Dish)을 추출한다.

Dish는 여러 개별 Recipe를 같은 음식 종류로 묶기 위한 표준 음식 개념이다.

규칙:
- 사람 이름, 홍보 문구, 조리 안내 표현은 제거한다.
- 재료 변형은 가능한 한 같은 음식으로 묶는다.
- 음식 자체의 정체성을 구성하는 단어는 제거하지 않는다.
- 지나치게 일반화하지 않는다.
- 제목만으로 판단하기 어렵다면 null을 반환한다.
- 설명은 출력하지 않는다.

출력 형식:

판단 가능한 경우:
{"dish": {"name": "표준 음식명"}}

판단할 수 없는 경우:
{"dish": null}
"""
    ),

    (
        "human",
        "제목: 너무 간단한데 맛있어서 놀라는 백종원 분식점 떡볶이 황금 레시피"
    ),
    (
        "assistant",
        '{"dish": {"name": "떡볶이"}}'
    ),

    (
        "human",
        "제목: 엄마의 레시피, 소고기 미역국 끓이는 법"
    ),
    (
        "assistant",
        '{"dish": {"name": "미역국"}}'
    ),

    (
        "human",
        "제목: 돼지고기 김치찌개 맛내는 비법"
    ),
    (
        "assistant",
        '{"dish": {"name": "김치찌개"}}'
    ),

    (
        "human",
        "제목: 백종원 오이무침 새콤달콤 맛있게~"
    ),
    (
        "assistant",
        '{"dish": {"name": "오이무침"}}'
    ),

    (
        "human",
        "제목: 오늘 저녁 뭐 먹지? 아이들도 너무 좋아해요"
    ),
    (
        "assistant",
        '{"dish": null}'
    ),

    (
        "human",
        "제목: {{ title }}"
    ),
], template_format="jinja2")

In [33]:
recipes = load_jsonl(
    "recipes_graph_prepared_v2_nonempty.jsonl"
)

title = recipes[0]["recipe"]["title"]

messages = few_prompt.invoke({
    "title": title
})

In [34]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL_NAME
)

In [23]:
from langchain_core.output_parsers import JsonOutputParser

chain = few_prompt | llm | JsonOutputParser()

In [35]:
title = recipes[0]["recipe"]["title"]

result = chain.invoke({
    "title": title
})

print(title)
print(result)

닭볶음탕 진짜진짜 황금레시피 알려 드려요~~^^
{'dish': '닭볶음탕'}


In [ ]:
import json
from pathlib import Path

OUTPUT_PATH = Path("../전처리/recipes_graph_with_dish.jsonl")

with OUTPUT_PATH.open("w", encoding="utf-8") as f:

    for i, row in enumerate(recipes[:5000]):

        title = row["recipe"]["title"]

        try:
            result = chain.invoke({
                "title": title
            })

            if result["dish"] is not None:
                row["dish"] = {
                    "name": result["dish"]
                }
            else:
                row["dish"] = None

        except Exception as e:
            print(f"{i}번째 오류:", title, e)
            row["dish"] = None

        f.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )

        if (i + 1) % 100 == 0:
            print(f"{i + 1}개 완료")

100개 완료
200개 완료
300개 완료
400개 완료
500개 완료
600개 완료
700개 완료
800개 완료
